In [1]:
!pip install psycopg2-binary

In [3]:
import os
import psycopg2
import pandas as pd

from datetime import datetime
from dotenv import dotenv_values
from helpers import get_token, search_for_artist, get_artist_top_tracks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import when, lit, col, concat

# Crear sesion de Spark

In [4]:
# Postgres and Redshift JDBCs
driver_path = "/home/coder/working_dir/driver_jdbc/postgresql-42.2.27.jre7.jar"

os.environ['PYSPARK_SUBMIT_ARGS'] = f'--driver-class-path {driver_path} --jars {driver_path} pyspark-shell'
os.environ['SPARK_CLASSPATH'] = driver_path

# Create SparkSession 
spark = SparkSession.builder \
        .master("local") \
        .appName("Conexion entre Pyspark y Redshift") \
        .config("spark.jars", driver_path) \
        .config("spark.executor.extraClassPath", driver_path) \
        .getOrCreate()

# Extract

Utilizando la Web API de Spotify. Obtenemos las caciones mas populares de un artista en un pais.

In [7]:
def extract():
    COUNTRY_CODES=["AR", "BR", "US", "MX"]
    ARTISTS_LIST=["Daft Punk", "Soda Stereo", "Arctic Monkeys", "The Strokes"]

    today = datetime.now()
    id_songs = []
    song_names = []
    artist_names = []
    album_names = []
    popularity_list = []
    duration_ms_list = []
    song_links = []
    country_code_list = []
    timestamps = []

    # Obtenemos el token de Spotify
    token = get_token()

    # Buscamos el id de cada artista
    id_artists = []
    songs = []
    
    for artist in ARTISTS_LIST:
        result = search_for_artist(token, artist_name=artist)
        id_artists.append(result["id"])

    # Obtenemos las canciones de cada artista   
    for country in COUNTRY_CODES:
        for id in id_artists:
            songs = get_artist_top_tracks(token, id_artist=id, country=country)
            # Guardamos la data en listas   
            for song in songs:
                id_songs.append(song["id"])
                song_names.append(song["name"])
                artist_names.append(song["artists"][0]["name"])
                album_names.append(song["album"]["name"])
                popularity_list.append(song["popularity"])
                duration_ms_list.append(song["duration_ms"])
                song_links.append(song["external_urls"]["spotify"])
                country_code_list.append(country)
                timestamps.append(today)

    # Creamos un diccionario de listas
    song_dict = {
        "id_song": id_songs,
        "song_name": song_names,
        "artist": artist_names,
        "album": album_names,
        "popularity": popularity_list,
        "duration_ms": duration_ms_list,
        "song_link": song_links,
        "country_code": country_code_list,
        "timestamp_": timestamps
    }

    return song_dict

# Transform

In [8]:
song_dict = extract()

In [10]:
# Creamos el DataFrame
df_songs = pd.DataFrame(song_dict)

my_schema = StructType([ StructField("id_song", StringType(), False)\
                      ,StructField("song_name", StringType(), False)\
                      ,StructField("artist", StringType(), False)\
                      ,StructField("album", StringType(), False)\
                      ,StructField("popularity", IntegerType(), False)\
                      ,StructField("duration_ms", IntegerType(), False)\
                      ,StructField("song_link", StringType(), False)\
                      ,StructField("country_code", StringType(), False)\
                      ,StructField("timestamp_", TimestampType(), False)\
                    ])
    
# Create the DataFrame with the specified column names
df = spark.createDataFrame(df_songs, schema=my_schema)
    
        
# Crear una columna personalizada para la clave primaria
df_to_write = df.withColumn("alternate_key", concat(col("id_song"), col("country_code")))

In [11]:
# Comprobar si el DataFrame está vacío
if df_to_write.rdd.isEmpty():
    raise Exception("The DataFrame is empty")

# Comprobar si la Primary Key es única
if not df_to_write.select("alternate_key").distinct().count() == df_to_write.count():
    raise Exception("A value from id is not unique")

In [38]:
df_to_write.printSchema()
df_to_write.show()

root
 |-- id_song: string (nullable = false)
 |-- song_name: string (nullable = false)
 |-- artist: string (nullable = false)
 |-- album: string (nullable = false)
 |-- popularity: integer (nullable = false)
 |-- duration_ms: integer (nullable = false)
 |-- song_link: string (nullable = false)
 |-- country_code: string (nullable = false)
 |-- timestamp_: timestamp (nullable = false)
 |-- alternate_key: string (nullable = false)

+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|             id_song|           song_name|     artist|               album|popularity|duration_ms|           song_link|country_code|          timestamp_|       alternate_key|
+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|5ruzrDWcT0vuJIOMW...|The Adults Are Ta

# Load

### Insertando datos en AWS Redshift

In [12]:
env = os.environ

In [20]:
# Conexion a AWS Redshift
conn = psycopg2.connect(
    host=env['REDSHIFT_HOST'],
    port=env['REDSHIFT_PORT'],
    dbname=env['REDSHIFT_DB'],
    user=env['REDSHIFT_USER'],
    password=env['REDSHIFT_PASSWORD']
)

In [ ]:
cursor = conn.cursor()
cursor.execute(f"""
create table if not exists {env['REDSHIFT_SCHEMA']}.popular_songs (
    id_song VARCHAR(250) NOT NULL,
    song_name VARCHAR(250) NOT NULL,
    artist VARCHAR(250) NOT NULL,
    album VARCHAR(150) NOT NULL,
    popularity INTEGER NOT NULL,
    duration_ms INTEGER NULL,
    song_link VARCHAR(250) NOT NULL,
    country_code VARCHAR(2) NOT NULL,
    timestamp_ TIMESTAMP NOT NULL,
    alternate_key VARCHAR(255) NOT NULL,
    PRIMARY KEY (alternate_key))
    distkey(artist)
    compound sortkey(timestamp_, country_code);
""")
conn.commit()
cursor.close()
print("Table created!")

In [21]:
# Cargamos los datos en la DB
df_to_write.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{env['REDSHIFT_HOST']}:{env['REDSHIFT_PORT']}/{env['REDSHIFT_DB']}") \
    .option("dbtable", f"{env['REDSHIFT_SCHEMA']}.popular_songs") \
    .option("user", env['REDSHIFT_USER']) \
    .option("password", env['REDSHIFT_PASSWORD']) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [22]:
# Leemos los datos desde la DB
# para controlar que se cargaron correctamente
query = f"select * from {env['REDSHIFT_SCHEMA']}.popular_songs"
data = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{env['REDSHIFT_HOST']}:{env['REDSHIFT_PORT']}/{env['REDSHIFT_DB']}") \
    .option("dbtable", f"({query}) as tmp_table") \
    .option("user", env['REDSHIFT_USER']) \
    .option("password", env['REDSHIFT_PASSWORD']) \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [23]:
data.printSchema()
data.show()

root
 |-- id_song: string (nullable = true)
 |-- song_name: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- album: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- song_link: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- timestamp_: timestamp (nullable = true)
 |-- alternate_key: string (nullable = true)

+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|             id_song|           song_name|     artist|               album|popularity|duration_ms|           song_link|country_code|          timestamp_|       alternate_key|
+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|2Foc5Q5nqNiosCNqt...|Get Lucky (Radio ...|  Daft

In [24]:
# Cerramos la conexion a la DB
conn.close()

### Conexion en local a PostgreSQL

In [15]:
# Conexion a AWS Redshift
conn = psycopg2.connect(
    host=env['POSTGRES_HOST'],
    port=env['POSTGRES_PORT'],
    dbname=env['POSTGRES_DB'],
    user=env['POSTGRES_USER'],
    password=env['POSTGRES_PASSWORD']
)

In [23]:
cursor = conn.cursor()
cursor.execute(f"""
create table if not exists {env['POSTGRES_SCHEMA']}.popular_songs (
    id_song VARCHAR(250) NOT NULL,
    song_name VARCHAR(250) NOT NULL,
    artist VARCHAR(250) NOT NULL,
    album VARCHAR(150) NOT NULL,
    popularity INTEGER NOT NULL,
    duration_ms INTEGER NULL,
    song_link VARCHAR(250) NOT NULL,
    country_code VARCHAR(2) NOT NULL,
    timestamp_ TIMESTAMP NOT NULL,
    alternate_key VARCHAR(255) NOT NULL,
    CONSTRAINT popular_songs_pk PRIMARY KEY(alternate_key)
    );
""")
conn.commit()
cursor.close()
print("Table created!")

Table created!


In [16]:
# Cargamos los datos en PostgreSQL
df_to_write.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{env['POSTGRES_HOST']}:{env['POSTGRES_PORT']}/{env['POSTGRES_DB']}") \
    .option("dbtable", f"{env['POSTGRES_SCHEMA']}.popular_songs") \
    .option("user", env['POSTGRES_USER']) \
    .option("password", env['POSTGRES_PASSWORD']) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [17]:
# Leemos los datos desde la DB
# para controlar que se cargaron correctamente
query = f"select * from {env['REDSHIFT_SCHEMA']}.popular_songs"
data = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{env['POSTGRES_HOST']}:{env['POSTGRES_PORT']}/{env['POSTGRES_DB']}") \
    .option("dbtable", f"{env['POSTGRES_SCHEMA']}.popular_songs") \
    .option("user", env['POSTGRES_USER']) \
    .option("password", env['POSTGRES_PASSWORD']) \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [18]:
data.printSchema()
data.show()

root
 |-- id_song: string (nullable = true)
 |-- song_name: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- album: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- song_link: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- timestamp_: timestamp (nullable = true)
 |-- alternate_key: string (nullable = true)

+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|             id_song|           song_name|     artist|               album|popularity|duration_ms|           song_link|country_code|          timestamp_|       alternate_key|
+--------------------+--------------------+-----------+--------------------+----------+-----------+--------------------+------------+--------------------+--------------------+
|2Foc5Q5nqNiosCNqt...|Get Lucky (Radio ...|  Daft

In [19]:
# Cerramos la conexion a la DB
conn.close()